In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

# 1. Crear directorio para guardar gráficos
os.makedirs('./charts', exist_ok=True)
sns.set_theme(style="whitegrid")

# 2. Cargar variables de entorno
load_dotenv()
API_URL = os.getenv('API_URL', 'https://data-charts-api.hexlet.app')
DATE_BEGIN = os.getenv('DATE_BEGIN', '2023-03-01')
DATE_END = os.getenv('DATE_END', '2023-09-01')

# 3. Consultar datos de la API
params = {'begin': DATE_BEGIN, 'end': DATE_END}
response_visits = requests.get(f'{API_URL}/visits', params=params)
response_regs = requests.get(f'{API_URL}/registrations', params=params)

df_api_visits = pd.DataFrame(response_visits.json())
df_api_regs = pd.DataFrame(response_regs.json())

# 4. Limpieza y preparación de visitas y registros
df_api_visits['datetime'] = pd.to_datetime(df_api_visits['datetime'])
df_api_regs['datetime'] = pd.to_datetime(df_api_regs['datetime'])

# Excluir bots
mask_no_bot = (
    ~df_api_visits['user_agent'].str.contains('bot', case=False, na=False) & 
    (df_api_visits['platform'] != 'bot')
)
df_visits_clean = df_api_visits[mask_no_bot].copy()
df_visits_clean = df_visits_clean.sort_values('datetime').drop_duplicates(subset=['visit_id'], keep='last')

df_visits_clean['date_group'] = df_visits_clean['datetime'].dt.normalize()
df_api_regs['date_group'] = df_api_regs['datetime'].dt.normalize()

# 5. Agrupar conversiones por fecha y plataforma
visits_grouped = df_visits_clean.groupby(['date_group', 'platform']).size().reset_index(name='visits')
regs_grouped = df_api_regs.groupby(['date_group', 'platform']).size().reset_index(name='registrations')

conversion_df = pd.merge(visits_grouped, regs_grouped, on=['date_group', 'platform'], how='inner')
conversion_df['conversion'] = (conversion_df['registrations'] / conversion_df['visits']) * 100
conversion_df = conversion_df.sort_values(by=['date_group', 'platform']).reset_index(drop=True)

# Guardar conversion.json
conversion_df.to_json("./conversion.json")

# 6. Integrar campañas publicitarias (ads.csv)
ads_df = pd.read_csv('./ads.csv')
ads_df['date'] = pd.to_datetime(ads_df['date'])
ads_df['date_group'] = ads_df['date'].dt.normalize()

ads_grouped = ads_df.groupby('date_group').agg({
    'cost': 'sum',
    'utm_campaign': 'first'
}).reset_index()

daily_conversion = conversion_df.groupby('date_group').agg({
    'visits': 'sum',
    'registrations': 'sum'
}).reset_index()

final_ads_df = pd.merge(daily_conversion, ads_grouped, on='date_group', how='left')
final_ads_df['cost'] = final_ads_df['cost'].fillna(0).astype(int)
final_ads_df['utm_campaign'] = final_ads_df['utm_campaign'].fillna('none')
final_ads_df = final_ads_df[['date_group', 'visits', 'registrations', 'cost', 'utm_campaign']].sort_values('date_group').reset_index(drop=True)

# Guardar ads.json
final_ads_df.to_json("./ads.json")

# 7. Generar gráficos y guardarlos en ./charts
conversion_df['date_str'] = conversion_df['date_group'].dt.strftime('%Y-%m-%d')
daily_summary = conversion_df.groupby('date_str').agg({
    'visits': 'sum',
    'registrations': 'sum',
    'date_group': 'first'
}).reset_index().sort_values('date_group')
daily_summary['conversion'] = (daily_summary['registrations'] / daily_summary['visits']) * 100

visits_pivot = conversion_df.pivot(index='date_str', columns='platform', values='visits').fillna(0)
regs_pivot = conversion_df.pivot(index='date_str', columns='platform', values='registrations').fillna(0)

# Gráfico 1: Visitas Totales
plt.figure(figsize=(16, 8))
ax = sns.barplot(data=daily_summary, x='date_str', y='visits', color='skyblue')
plt.title('Total Visits', fontsize=14)
plt.xlabel('date_group', fontsize=12)
plt.ylabel('visits', fontsize=12)
plt.xticks(rotation=45, ha='right')
for p in ax.patches:
    val = int(p.get_height())
    if val > 0:
        ax.annotate(str(val), (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=10, color='black', xytext=(0, 3), textcoords='offset points')
plt.tight_layout()
plt.savefig('./charts/total_visits.png', dpi=300)
plt.close()

# Gráfico 2: Visitas por Plataforma (Stacked)
visits_pivot.plot(kind='bar', stacked=True, figsize=(16, 8), width=0.8)
plt.title('Visits by Platform (Stacked)', fontsize=14)
plt.xlabel('date_group', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(title='platform')
plt.tight_layout()
plt.savefig('./charts/total_visits_by_platform.png', dpi=300)
plt.close()

# Gráfico 3: Registros Totales
plt.figure(figsize=(16, 8))
ax = sns.barplot(data=daily_summary, x='date_str', y='registrations', color='skyblue')
plt.title('Total Weekly Registrations', fontsize=14)
plt.xlabel('date_group', fontsize=12)
plt.ylabel('registrations', fontsize=12)
plt.xticks(rotation=45, ha='right')
for p in ax.patches:
    val = int(p.get_height())
    if val > 0:
        ax.annotate(str(val), (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=10, color='black', xytext=(0, 3), textcoords='offset points')
plt.tight_layout()
plt.savefig('./charts/total_registrations.png', dpi=300)
plt.close()

# Gráfico 4: Registros por Plataforma (Stacked)
regs_pivot.plot(kind='bar', stacked=True, figsize=(16, 8), width=0.8)
plt.title('Weekly Registrations by Platform (Stacked)', fontsize=14)
plt.xlabel('date_group', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(title='platform')
plt.tight_layout()
plt.savefig('./charts/total_registrations_by_platform.png', dpi=300)
plt.close()

# Gráfico 5: Conversión por Plataforma
platforms = ['android', 'ios', 'web']
fig, axes = plt.subplots(3, 1, figsize=(14, 14), sharex=True)
for i, plat in enumerate(platforms):
    plat_data = conversion_df[conversion_df['platform'] == plat].copy().sort_values('date_group')
    axes[i].plot(plat_data['date_str'], plat_data['conversion'], marker='o', label=plat)
    axes[i].set_title(f'Conversion {plat}', fontsize=13)
    axes[i].set_ylabel('Conversion (%)', fontsize=11)
    axes[i].legend(loc='center left' if plat == 'android' else 'center right')
    for x_idx, (idx_row, row) in enumerate(plat_data.iterrows()):
        axes[i].annotate(f"{int(round(row['conversion']))}%", 
                         (x_idx, row['conversion']),
                         textcoords="offset points", xytext=(0, 6), ha='center', fontsize=9)
plt.xlabel('Date', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('./charts/conversion_by_platforms.png', dpi=300)
plt.close()

# Gráfico 6: Conversión General
plt.figure(figsize=(14, 7))
plt.plot(daily_summary['date_str'], daily_summary['conversion'], marker='o', label='Overall Conversion')
plt.title('Overall Conversion', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Conversion (%)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend()
for x_idx, val in enumerate(daily_summary['conversion']):
    plt.annotate(f"{int(round(val))}%", (x_idx, val),
                 textcoords="offset points", xytext=(0, 6), ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('./charts/conversion.png', dpi=300)
plt.close()

# Gráfico 7: Costos de Campañas
ads_costs = ads_df.groupby('date_group')['cost'].sum().reset_index().sort_values('date_group')
ads_costs['date_str'] = ads_costs['date_group'].dt.strftime('%Y-%m-%d')
plt.figure(figsize=(14, 7))
plt.plot(ads_costs['date_str'], ads_costs['cost'], marker='o')
plt.title('Aggregated Ad Campaign Costs (by day)', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Cost (RUB)', fontsize=12)
plt.xticks(rotation=45, ha='right')
for x_idx, val in enumerate(ads_costs['cost']):
    plt.annotate(f"{val} RUB", (x_idx, val),
                 textcoords="offset points", xytext=(0, 6), ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('./charts/ads_cost.png', dpi=300)
plt.close()

# Gráfico 8: Actividad durante Campañas
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 14), sharex=True)
campaign_groups = ads_df.groupby('utm_campaign').agg(
    source=('utm_source', 'first'),
    medium=('utm_medium', 'first'),
    start=('date_group', 'min'),
    end=('date_group', 'max')
).reset_index()
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

ax1.plot(daily_summary['date_group'], daily_summary['visits'], marker='o', color='black', label='Visits')
ax1.axhline(daily_summary['visits'].mean(), color='grey', linestyle='--', label='Average Number of Visits')
ax1.set_title('Visits during marketing active days', fontsize=13)
ax1.set_ylabel('Unique Visits', fontsize=11)

ax2.plot(daily_summary['date_group'], daily_summary['registrations'], marker='o', color='green', label='Registrations')
ax2.axhline(daily_summary['registrations'].mean(), color='grey', linestyle='--', label='Average Number of Registration')
ax2.set_title('Registrations during marketing active days', fontsize=13)
ax2.set_ylabel('Unique Users', fontsize=11)

for idx, row in campaign_groups.iterrows():
    c_label = f"{row['source']} {row['medium']} {row['utm_campaign']}"
    col = colors[idx % len(colors)]
    ax1.axvspan(row['start'], row['end'], alpha=0.45, color=col, label=c_label)
    ax2.axvspan(row['start'], row['end'], alpha=0.45, color=col, label=c_label)

ax1.legend(loc='lower left')
ax2.legend(loc='upper right')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('./charts/activity_during_marketing_campaign.png', dpi=300)
plt.close()

print("¡Listo! Todo ejecutado y guardado correctamente.")